In [ ]:
"""
Лабораторная работа: Научные вычисления и визуализация (NumPy, SciPy, Matplotlib)

Цель работы
Освоить инструменты обработки и визуализации биомедицинских сигналов на Python. Научиться векторизовать вычисления (NumPy) для ускорения работы с большими данными, применять методы аппроксимации, поиска корней и численного интегрирования (SciPy), а также создавать сложные аналитические дашборды с многокомпонентными графиками (Matplotlib).

Постановка задачи
В лаборатории биомеханики проводится анализ движений пациента, проходящего реабилитацию после травмы плечевого сустава. На запястье пациента закреплен оптический маркер системы Motion Capture (захват движений). Пациент выполняет упражнение: плавное поднятие руки вперед и вверх с последующим опусканием.

Современные системы MoCap генерируют огромные массивы данных (сотни кадров в секунду), поэтому для анализа требуется использовать высокопроизводительные инструменты. Кроме того, «сырые» данные координат (X, Y, Z) содержат шум датчиков, а сама система координат сенсора изначально не выровнена с глобальной системой координат лаборатории.

Необходимо разработать скрипт, который:

Проведет предварительный бенчмаркинг, доказав эффективность библиотеки NumPy по сравнению со стандартным Python при обработке больших массивов (1 млн точек).
Выровняет систему координат датчика с лабораторией (используя матрицу поворота).
Очистит вертикальную траекторию (Z) от шума с помощью полиномиальной аппроксимации.
Рассчитает биомеханические параметры: скорости, ускорения и суммарный пройденный путь кисти в 3D-пространстве (через интеграл модуля вектора скорости).
Найдет точное время достижения максимальной высоты подъема руки и время завершения движения (пересечение базовой плоскости).
Построит аналитический дашборд биомеханики из 4-х графиков.
Математическая модель и этапы работы

0. Бенчмаркинг производительности (NumPy vs Pure Python)
Прежде чем анализировать конкретное упражнение, докажите эффективность выбранного инструмента.

Напишите функцию, которая генерирует траекторию из 1 000 000 точек.
Рассчитайте массив расстояний между каждыми двумя соседними точками траектории в 3D-пространстве по формуле: L = sqrt((X[i+1] - X[i])^2 + (Y[i+1] - Y[i])^2 + (Z[i+1] - Z[i])^2)
Реализуйте этот расчет двумя способами:
Используя стандартные списки Python (list), цикл for и модуль math.
Используя массивы numpy без циклов (применяя срезы [1:] - [:-1] или функцию np.diff).
Замерьте время выполнения обоих алгоритмов с помощью модуля time и выведите в консоль ускорение (во сколько раз NumPy отработал быстрее).
1. Генерация и Линейная алгебра (NumPy)

Генерация идеальной траектории: Время T от 0 до 6.5 секунд (500 точек). x(t) = 1.5 * sin(t) y(t) = 2.0 * cos(t) z(t) = -0.4 * t^2 + 2.5 * t + 0.5 (полином, описывающий подъем и опускание руки)
Шум: к оси Z добавлен нормальный (гауссовский) шум.
Поворот осей: сенсор смещен на угол 30 градусов относительно лаборатории в плоскости XY. Необходимо применить матрицу поворота 2x2 к координатам X и Y (используйте матричное умножение).
2. Аппроксимация и уравнения (SciPy)

Аппроксимировать зашумленный сигнал Z(t) с помощью scipy.optimize.curve_fit (модель — полином 2-й степени).
Вывести функцию производной для аппроксимированной функции Z(t).
Используя scipy.optimize.root_scalar, найти корень производной — время максимального подъема руки.
Найти корень самой функции Z(t) на интервале падения — точное время, когда рука опускается ниже уровня датчика (завершение движения).
3. Кинематика и математический анализ (NumPy / SciPy)

Вычислить проекции скорости (Vx, Vy, Vz), используя численное дифференцирование np.gradient.
Рассчитать модуль вектора 3D-скорости для каждой точки: V = sqrt(Vx^2 + Vy^2 + Vz^2).
Рассчитать вертикальное ускорение Az.
Вычислить суммарную длину траектории (пройденный путь кисти L), проинтегрировав модуль 3D-скорости по времени, используя метод Симпсона scipy.integrate.simpson.
4. Визуализация (Matplotlib)
Запрещено использовать стандартные стили "из коробки" без изменений. Требуется самостоятельно настроить цвета, подписи и сетки.
Необходимо создать фигуру размером 2x2:

График 1: 3D-траектория (projection='3d'). Зашумленные исходные данные отобразить полупрозрачными точками, а сглаженную траекторию — контрастной жирной линией.
График 2: Профиль высоты Z(t). Отметить полупрозрачной закраской (fill_between) зону от Z=0 до графика высоты. Проставить текстовые аннотации со стрелками (annotate) на точках максимума и окончания движения.
График 3: Вертикальная скорость Vz(t) и ускорение Az(t). Реализовать на одном графике с использованием двух независимых осей Y (twinx()). Обязательно наличие общей легенды.
График 4: 2D-проекция движения кисти в плоскости XY (scatter). Цвет точек должен градиентно зависеть от текущего модуля 3D-скорости V в данный момент времени. Использовать colormap и добавить цветовую шкалу colorbar.
Требования к реализации:

Строгий и полный отказ от циклов (for, while) для любых математических операций над массивами данных (за исключением Этапа 0). Вся работа с данными должна быть векторизована через NumPy. """

In [ ]:
"""
Лабораторная работа: NumPy, SciPy, Matplotlib
Тема: Биомеханический анализ движения пациента (Motion Capture)

Ваша задача: заполнить все пропуски (TODO) и реализовать логику.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, root_scalar
from scipy.integrate import simpson
import time
import math

# ==================== 0. БЕНЧМАРКИНГ (NUMPY vs PYTHON) ====================

def benchmark_performance(N: int = 1_000_000):
    """Сравнение скорости расчетов стандартного Python и NumPy на больших данных."""
    print(f"--- ЗАПУСК БЕНЧМАРКА НА {N} ТОЧЕК ---")
    
    # Генерация данных (используем NumPy для скорости генерации, затем конвертируем в списки)
    t_arr = np.linspace(0, 10, N)
    x_arr = np.sin(t_arr)
    y_arr = np.cos(t_arr)
    z_arr = t_arr ** 2

    # Конвертация в обычные списки Python
    x_list = x_arr.tolist()
    y_list = y_arr.tolist()
    z_list = z_arr.tolist()

    # --- Метод 1: Pure Python ---
    start_time = time.perf_counter()
    distances_list = []
    # TODO: Вычислите расстояния между соседними точками 3D-траектории, используя цикл for.
    # Формула: sqrt((x[i+1]-x[i])^2 + (y[i+1]-y[i])^2 + (z[i+1]-z[i])^2)
    # Используйте функцию math.sqrt
    
    # for i in range(...):
    #     dist = ...
    #     distances_list.append(dist)
    
    python_time = time.perf_counter() - start_time
    print(f"Pure Python (цикл for): {python_time:.4f} секунд")

    # --- Метод 2: NumPy ---
    start_time = time.perf_counter()
    # TODO: Вычислите те же расстояния, но используя только векторизацию NumPy (БЕЗ ЦИКЛОВ).
    # Подсказка 1: Можно использовать функцию np.diff(), которая сама считает разницу соседних элементов.
    # Подсказка 2: Или использовать срезы массивов: dx = x_arr[1:] - x_arr[:-1]
    
    distances_arr = ... # ЗАМЕНИТЕ ЭТО
    
    numpy_time = time.perf_counter() - start_time
    print(f"NumPy (векторизация):   {numpy_time:.4f} секунд")
    
    # TODO: Рассчитайте и выведите, во сколько раз NumPy оказался быстрее.
    # speedup = ...
    # print(f"NumPy быстрее в {speedup:.1f} раз!\n")

# ==================== 1. ПОДГОТОВКА И ЛИНЕЙНАЯ АЛГЕБРА ====================

def generate_and_transform_mocap_data(num_points: int = 500) -> tuple:
    """Генерация сырых данных и выравнивание систем координат."""
    t = np.linspace(0, 6.5, num_points)
    
    x_ideal = 1.5 * np.sin(t)
    y_ideal = 2.0 * np.cos(t)
    z_ideal = -0.4 * t**2 + 2.5 * t + 0.5
    
    # TODO: Добавьте случайный гауссовский шум (mean=0, std=0.2) к массиву z_ideal
    z_noisy = ... 

    # TODO: Сенсор повернут на 30 градусов (pi/6). Сформируйте матрицу поворота 2x2.
    # Примените умножение матриц для X и Y.
    theta = ...
    rotation_matrix = ...
    
    x_rot = ...
    y_rot = ...

    return t, x_rot, y_rot, z_noisy

# ==================== 2. АППРОКСИМАЦИЯ И ПОИСК КОРНЕЙ ====================

def polynomial_2nd_degree(t, a, b, c):
    """Модель для аппроксимации высоты."""
    return a * t**2 + b * t + c

def polynomial_derivative(t, a, b):
    """Производная полинома 2-й степени."""
    return 2 * a * t + b

def analyze_biomechanics(t: np.ndarray, z_noisy: np.ndarray):
    """Сглаживание и поиск ключевых фаз движения."""
    
    # TODO: Используйте curve_fit для нахождения коэффициентов a, b, c
    popt, pcov = ... # (замените)
    
    # TODO: Вычислите сглаженную высоту
    z_smooth = ...
    
    # TODO: Найти время t_max (корень polynomial_derivative через root_scalar)
    t_max = ... 
    z_max = ... # высота в момент t_max
    
    # TODO: Найти время завершения упражнения t_end (корень polynomial_2nd_degree на спаде)
    t_end = ...

    return z_smooth, t_max, z_max, t_end

# ==================== 3. МАТАН: КИНЕМАТИКА ====================

def calculate_kinematics(t, x, y, z_smooth) -> tuple:
    """Вычисление скоростей, ускорения и длины пути."""
    
    # TODO: Вычислите массивы скоростей по осям (np.gradient)
    vx = ...
    vy = ...
    vz = ...
    
    # TODO: Вычислите модуль вектора 3D-скорости: V = sqrt(vx^2 + vy^2 + vz^2)
    v_total = ...
    
    # TODO: Вычислите вертикальное ускорение az (производная от vz)
    az = ...
    
    # TODO: Рассчитайте общую длину 3D-траектории (интеграл модуля скорости через simpson)
    total_distance = ...
    
    return vz, az, v_total, total_distance

# ==================== 4. СЛОЖНАЯ ВИЗУАЛИЗАЦИЯ ====================

def build_dashboard(t, x, y, z_noisy, z_smooth, vz, az, v_total, t_max, z_max, t_end, total_distance):
    """Построение биомеханической панели из 4 графиков."""
    
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Анализ Motion Capture (Суммарный путь кисти: {total_distance:.2f} м)", fontsize=16)
    
    # --- График 1: 3D Траектория ---
    ax1 = fig.add_subplot(2, 2, 1, projection='3d')
    # TODO: Постройте 3D scatter (x, y, z_noisy) и 3D plot (x, y, z_smooth)

    # --- График 2: Профиль высоты Z(t) ---
    ax2 = fig.add_subplot(2, 2, 2)
    # TODO: Постройте график z_smooth(t), используйте fill_between, scatter и annotate для максимума и конца

    # --- График 3: Скорость и Ускорение (2 оси Y) ---
    ax3 = fig.add_subplot(2, 2, 3)
    # TODO: График vz(t) на ax3. Создайте ax3_twin = ax3.twinx() и постройте az(t). Добавьте легенду.

    # --- График 4: Плоскость XY и цветовое кодирование ---
    ax4 = fig.add_subplot(2, 2, 4)
    # TODO: Постройте scatter(x, y, c=v_total, cmap='viridis'). Добавьте fig.colorbar(..., ax=ax4).
    
    plt.tight_layout()
    # TODO: Отобразите график
    pass

# ==================== ГЛАВНЫЙ ЦИКЛ ====================

def main():
    # Этап 0: Демонстрация мощи NumPy
    benchmark_performance()
    
    print("=== БИОМЕХАНИЧЕСКИЙ АНАЛИЗАТОР ЗАПУЩЕН ===")
    # 1. Получение данных
    t, x, y, z_noisy = generate_and_transform_mocap_data()
    
    # 2. Математическое моделирование
    z_smooth, t_max, z_max, t_end = analyze_biomechanics(t, z_noisy)
    print(f"Максимальная высота кисти: {z_max:.2f} м (время: {t_max:.2f} с)")
    print(f"Время завершения движения: {t_end:.2f} с")
    
    # Отсекаем данные после завершения упражнения
    valid_idx = t <= t_end
    t, x, y = t[valid_idx], x[valid_idx], y[valid_idx]
    z_noisy, z_smooth = z_noisy[valid_idx], z_smooth[valid_idx]
    
    # 3. Расчет кинематики
    vz, az, v_total, total_dist = calculate_kinematics(t, x, y, z_smooth)
    print(f"Пройденная длина 3D-траектории: {total_dist:.2f} м")
    
    # 4. Визуализация
    print("Отрисовка дашборда...")
    build_dashboard(t, x, y, z_noisy, z_smooth, vz, az, v_total, t_max, z_max, t_end, total_dist)

if __name__ == "__main__":
    main()
